In [1]:
"""
download_ccc_reports.py
------------------------
Downloads Climate Change Committee (CCC) Progress Report PDFs for the
RCCB Index project. The CCC's website (theccc.org.uk) has NO public API
(confirmed by inspection -- it's a standard WordPress site), so this
script does real HTML scraping of the listing pages instead of querying
a structured endpoint.

WHY THIS MATTERS FOR YOUR METHODOLOGY CHAPTER:
Same "justify the corpus" principle as the other two scripts, adapted to
a source with no API: the inclusion rule here is "every report linked
from the CCC's own canonical category listing page", not a hand-picked
selection. The category page IS the CCC's own definition of what counts
as a "Progress Report (Net Zero)" -- using it as the source of truth is
more defensible than guessing at titles yourself.

INCLUSION RULE (edit LISTING_PAGES below to change this):
    A document is included if it is linked from one of the CCC's own
    publication-category listing pages:
      - Progress reports (Net Zero)   [primary target per your proposal]
      - Carbon budgets                [optional -- related advice reports]
    Pagination on each listing page is followed until no further pages
    are found, so ALL reports in that category are captured, not just
    the first page shown by default.

HOW THIS WORKS (since there's no API):
    1. Fetch each listing page's HTML.
    2. Parse out every link matching the CCC's publication URL pattern
       (theccc.org.uk/publication/<slug>/).
    3. Visit each publication's own page, and extract the actual PDF
       download link from it (PDFs are hosted at
       theccc.org.uk/wp-content/uploads/<year>/<month>/<file>.pdf and
       are NOT predictable from the title alone, so this step is
       required rather than guessed).
    4. Download each PDF found.

USAGE:
    pip install requests beautifulsoup4
    python download_ccc_reports.py

OUTPUT:
    ccc_raw/                 <- one .pdf file per matched report
    ccc_metadata.csv          <- full audit trail: report title, listing
                                  page it came from, publication page
                                  URL, PDF URL, and download status.

NOTE ON DEPENDENCIES:
    This script uses BeautifulSoup (bs4) for HTML parsing, since CCC's
    page structure is not as simple as a CSV/XML feed and regex-only
    HTML parsing is fragile. Install with:
        pip install beautifulsoup4
"""

import csv
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

# -----------------------------------------------------------------------
# 1. CONFIGURATION
# -----------------------------------------------------------------------

OUTPUT_DIR = Path("ccc_raw")
METADATA_FILE = Path("ccc_metadata.csv")

BASE_URL = "https://www.theccc.org.uk"

REQUEST_TIMEOUT = 30
RETRY_ATTEMPTS = 3
RETRY_BACKOFF = 5
POLITE_DELAY = 1.5
MAX_LISTING_PAGES = 15  # safety cap per category

HEADERS = {
    "User-Agent": "NTU-MSc-Research-RCCB-Index/1.0 (academic research; contact via university email)"
}

# -----------------------------------------------------------------------
# 2. LISTING PAGES TO SCRAPE -- THIS IS YOUR JUSTIFICATION
#
# These are the CCC's OWN category pages, found by inspecting the site's
# publications navigation. Using their categorisation (rather than a
# keyword search) means the inclusion rule is "whatever the CCC itself
# classifies as a Progress Report", which is the most defensible
# definition available for this source.
# -----------------------------------------------------------------------

LISTING_PAGES = [
    {
        "category": "Progress reports (Net Zero)",
        "url": "{}/publicationtype/report/progress-reports-net-zero/".format(BASE_URL),
    },
    # Uncomment to also include Carbon Budget advice reports:
    # {
    #     "category": "Carbon budgets",
    #     "url": "{}/publicationtype/advice/carbon-budgets/".format(BASE_URL),
    # },
]

# Study window, matching the other two scripts.
START_YEAR = 2008
END_YEAR = 2026


# -----------------------------------------------------------------------
# 3. FETCH HELPERS
# -----------------------------------------------------------------------

def fetch_html(url):
    """Fetch a URL's HTML with retries. Returns text or None."""
    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            if resp.status_code == 200:
                return resp.text
            print("    [WARN] {} -> HTTP {} (attempt {}/{})".format(url, resp.status_code, attempt, RETRY_ATTEMPTS))
        except requests.RequestException as exc:
            print("    [ERROR] {} -> {} (attempt {}/{})".format(url, exc, attempt, RETRY_ATTEMPTS))
        if attempt < RETRY_ATTEMPTS:
            time.sleep(RETRY_BACKOFF * attempt)
    return None


# -----------------------------------------------------------------------
# 4. STEP 1: discover publication page links from listing pages
# -----------------------------------------------------------------------

def find_publication_links(listing_html):
    """
    Parse a listing page's HTML and return all unique links matching
    the CCC's publication URL pattern: /publication/<slug>/
    """
    soup = BeautifulSoup(listing_html, "html.parser")
    links = {}
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/publication/" in href:
            full_url = urljoin(BASE_URL, href)
            title = a.get_text(strip=True) or "UNKNOWN TITLE"
            if full_url not in links or len(title) > len(links[full_url]):
                links[full_url] = title
    return links


def find_next_page_url(listing_html, current_url):
    """
    Look for a 'next page' pagination link in the listing page HTML.
    Returns the next page URL, or None if there isn't one.
    WordPress pagination commonly uses rel="next" or a '.next' class --
    this checks both, falling back to None if neither is found (i.e.
    treat absence of a recognised 'next' link as the last page).
    """
    soup = BeautifulSoup(listing_html, "html.parser")

    next_link = soup.find("a", rel="next")
    if next_link and next_link.get("href"):
        return urljoin(current_url, next_link["href"])

    next_link = soup.find("a", class_=re.compile(r"\bnext\b"))
    if next_link and next_link.get("href"):
        return urljoin(current_url, next_link["href"])

    return None


def discover_publication_pages():
    """
    Walk every configured listing page (with pagination) and return a
    de-duplicated dict of {publication_url: {"title":..., "category":...}}.
    """
    all_pubs = {}

    for listing in LISTING_PAGES:
        category = listing["category"]
        url = listing["url"]
        page_num = 1

        print("Scraping listing: {} ({})".format(category, url))

        while url and page_num <= MAX_LISTING_PAGES:
            html = fetch_html(url)
            time.sleep(POLITE_DELAY)
            if html is None:
                break

            links = find_publication_links(html)
            new_count = 0
            for pub_url, title in links.items():
                if pub_url not in all_pubs:
                    all_pubs[pub_url] = {"title": title, "category": category}
                    new_count += 1

            print("    page {}: {} links found ({} new)".format(page_num, len(links), new_count))

            if new_count == 0 and page_num > 1:
                break  # likely looping or exhausted

            next_url = find_next_page_url(html, url)
            if next_url == url:
                break
            url = next_url
            page_num += 1

    return all_pubs


# -----------------------------------------------------------------------
# 5. STEP 2: resolve each publication page to its PDF link
# -----------------------------------------------------------------------

def find_pdf_link(pub_html, pub_url):
    """
    Parse a publication detail page and find its PDF download link.
    CCC publication pages typically link to a file under
    /wp-content/uploads/... ending in .pdf.
    """
    soup = BeautifulSoup(pub_html, "html.parser")
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.lower().endswith(".pdf"):
            return urljoin(pub_url, href)
    return None


def extract_publication_date(pub_html):
    """
    Best-effort extraction of a publication date from the page, used to
    filter by study year range. Looks for a <time> tag (common in
    WordPress themes) first, falls back to empty string if not found --
    rows with no detected date are NOT auto-excluded, they're flagged
    for manual review instead.
    """
    soup = BeautifulSoup(pub_html, "html.parser")
    time_tag = soup.find("time")
    if time_tag:
        return time_tag.get("datetime", time_tag.get_text(strip=True))
    return ""


# -----------------------------------------------------------------------
# 6. DOWNLOAD
# -----------------------------------------------------------------------

def slugify(url):
    return re.sub(r"[^a-zA-Z0-9_-]", "_", url.rstrip("/").split("/")[-1])


def download_pdf(pdf_url, doc_id):
    out_file = OUTPUT_DIR / "{}.pdf".format(doc_id)
    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            resp = requests.get(pdf_url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            if resp.status_code == 200 and resp.content:
                out_file.write_bytes(resp.content)
                print("  [OK] {} ({:,} bytes)".format(out_file.name, len(resp.content)))
                return {"status": "success", "http_code": resp.status_code, "file": str(out_file)}
            print("  [WARN] {}: HTTP {} (attempt {}/{})".format(pdf_url, resp.status_code, attempt, RETRY_ATTEMPTS))
        except requests.RequestException as exc:
            print("  [ERROR] {}: {} (attempt {}/{})".format(pdf_url, exc, attempt, RETRY_ATTEMPTS))
        if attempt < RETRY_ATTEMPTS:
            time.sleep(RETRY_BACKOFF * attempt)
    return {"status": "FAILED", "http_code": "", "file": ""}


# -----------------------------------------------------------------------
# 7. MAIN
# -----------------------------------------------------------------------

def main():
    OUTPUT_DIR.mkdir(exist_ok=True)

    print("=" * 70)
    print("STEP 1: Discovering publication pages from CCC listing pages")
    print("=" * 70)
    publications = discover_publication_pages()
    print("\nDiscovered {} unique publication pages.\n".format(len(publications)))

    print("=" * 70)
    print("STEP 2: Resolving each publication page to its PDF, and downloading")
    print("=" * 70)
    results = []
    for pub_url, meta in publications.items():
        doc_id = slugify(pub_url)
        print("Resolving: {} ({})".format(meta["title"], pub_url))

        pub_html = fetch_html(pub_url)
        time.sleep(POLITE_DELAY)

        row = {
            "doc_id": doc_id,
            "title": meta["title"],
            "category": meta["category"],
            "publication_url": pub_url,
            "publication_date": "",
            "pdf_url": "",
            "status": "",
            "http_code": "",
            "file": "",
        }

        if pub_html is None:
            row["status"] = "PAGE_FETCH_FAILED"
            results.append(row)
            continue

        row["publication_date"] = extract_publication_date(pub_html)
        pdf_url = find_pdf_link(pub_html, pub_url)

        if not pdf_url:
            print("  [WARN] no PDF link found on this publication page")
            row["status"] = "NO_PDF_FOUND"
            results.append(row)
            continue

        row["pdf_url"] = pdf_url
        dl_result = download_pdf(pdf_url, doc_id)
        time.sleep(POLITE_DELAY)
        row.update(dl_result)
        results.append(row)

    with open(METADATA_FILE, "w", newline="", encoding="utf-8") as f:
        fieldnames = ["doc_id", "title", "category", "publication_url", "publication_date",
                      "pdf_url", "status", "http_code", "file"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)

    succeeded = sum(1 for r in results if r["status"] == "success")
    no_pdf = sum(1 for r in results if r["status"] == "NO_PDF_FOUND")
    failed = len(results) - succeeded - no_pdf
    print("\nDone. {} downloaded, {} had no PDF found, {} failed (of {} rows).".format(
        succeeded, no_pdf, failed, len(results)))
    print("Full audit trail written to {}.".format(METADATA_FILE))
    print("  - Filter 'publication_date' to your study window (2008-2024) manually --")
    print("    date extraction here is best-effort; verify before treating as final.")
    print("  - Review NO_PDF_FOUND rows: a small number of CCC pages link to external")
    print("    hosts or present content as a web page rather than a PDF.")


if __name__ == "__main__":
    main()


# -----------------------------------------------------------------------
# KEEPING THIS CURRENT -- same caveats as the other two scripts
# -----------------------------------------------------------------------
# 1. Re-running this script will pick up newly published CCC reports
#    automatically, as long as the CCC adds them to the same listing
#    page category used here.
# 2. This is HTML scraping, not an API -- it is the most fragile of the
#    three scripts. If CCC redesigns their site, the link-pattern checks
#    in find_publication_links() and find_pdf_link() may need updating.
#    Treat failures here as a signal to manually inspect the site, not
#    necessarily a sign your script is broken.
# 3. Web scraping is generally permitted for academic research under UK
#    law (no login/paywall bypass here, robots.txt should still be
#    checked), but always check the target site's terms of use before
#    scraping at scale, and keep the POLITE_DELAY between requests.

STEP 1: Discovering publication pages from CCC listing pages
Scraping listing: Progress reports (Net Zero) (https://www.theccc.org.uk/publicationtype/report/progress-reports-net-zero/)
    page 1: 12 links found (12 new)
    page 2: 10 links found (10 new)

Discovered 22 unique publication pages.

STEP 2: Resolving each publication page to its PDF, and downloading
Resolving: 24 June 2026Progress in reducing emissions 
2026 report to ParliamentRead more (https://www.theccc.org.uk/publication/progress-in-reducing-emissions-2026-report-to-parliament/)
  [OK] progress-in-reducing-emissions-2026-report-to-parliament.pdf (83,647,501 bytes)
Resolving: 25 February 2026Progress in reducing emissions in Scotland – 2025 report to ParliamentRead more (https://www.theccc.org.uk/publication/progress-in-reducing-emissions-in-scotland-2025/)
  [OK] progress-in-reducing-emissions-in-scotland-2025.pdf (10,799,253 bytes)
Resolving: 25 June 2025Progress in reducing emissions – 2025 report to ParliamentRea